# Clase 5 · 01 · Primer run: qué registra MLflow

En MLflow cada entrenamiento es un **run** dentro de un **experimento**. Un run guarda:

| Qué | Ejemplos | Para qué |
|---|---|---|
| **Parámetros** | modelo, features, hiperparámetros, umbral | Saber *cómo* se entrenó |
| **Métricas** | PR-AUC, precision, recall, accuracy | Saber *qué tan bien* anduvo |
| **Artefactos** | gráficos, CSV, el modelo serializado | Revisar resultados y reutilizar el modelo |
| **Tags** | alumno, notebook, URI del modelo | Buscar y filtrar runs |

Vamos a hacer un run "a mano", mirarlo en la interfaz, compararlo con lo que registra **autolog**, comprobar que es reproducible y volver a cargar el modelo.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/ml_lab

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
spark.conf.set("spark.sql.session.timeZone", "UTC")
T = lambda name: qualified_table(config, name)
client = MlflowClient()

In [ ]:
assert spark.catalog.tableExists(T("ml_features")), "Falta ml_features. Ejecutá primero 00_preflight."
experiment = setup_experiment(spark, config)
data_version = table_version(spark, T("ml_features"))
print("Experimento:", experiment.name)

## 1. Los datos y una línea de base

Antes de entrenar nada: ¿qué accuracy tendría un "modelo" que **siempre** dice "no hay contracargo"?

In [ ]:
train = load_split(spark, config, "train")
test = load_split(spark, config, "test")
rate = test["chargeback"].mean()
print(f"train: {len(train):,} filas · test: {len(test):,} filas")
print(f"Contracargos en test: {test['chargeback'].sum():,} ({rate:.1%})")
print(f"Accuracy de decir siempre 'no': {1 - rate:.1%}  ← sin detectar ni un solo contracargo")

Con clases desbalanceadas la **accuracy engaña**. Por eso en toda la práctica la métrica principal es el **PR-AUC** (área bajo la curva precision-recall): resume qué tan bien el modelo ordena las transacciones de más a menos riesgosas, sin depender de un umbral. Un modelo al azar tiene un PR-AUC igual a la tasa de contracargos.

## 2. Un run a mano

Entrenamos una regresión logística con dos variables (`amount` y `payment_channel`) y registramos todo con las funciones básicas de MLflow. Leé el código: es lo único que hace falta para usar MLflow en cualquier proyecto.

In [ ]:
features = FEATURE_SETS["basicas"]
threshold = 0.5

with mlflow.start_run(run_name="01_manual_logistica", tags={**base_tags(config, "01"), "tipo": "modelo"}) as run:
    # 1) Parámetros: cómo se entrena
    mlflow.log_params({"model_type": "logistic_regression", "feature_set": "basicas", "C": 1.0,
                       "class_weight": "ninguno", "threshold": threshold,
                       "data_table": "ml_features", "data_version": data_version})

    # 2) Entrenamiento (preprocesamiento + modelo en un único Pipeline de scikit-learn)
    pipeline = build_pipeline("logistic_regression", features, C=1.0)
    pipeline.fit(train[features], train["chargeback"])

    # 3) Métricas: qué tan bien anduvo sobre datos que no vio
    scores = pipeline.predict_proba(test[features])[:, 1]
    metrics = evaluate(scores, test["chargeback"], threshold, "test")
    metrics["train_pr_auc"] = average_precision_score(train["chargeback"], pipeline.predict_proba(train[features])[:, 1])
    mlflow.log_metrics(metrics)

    # 4) Artefactos: un gráfico y el modelo con su firma
    mlflow.log_figure(confusion_figure(test["chargeback"], scores, threshold, "Logística · básicas"), "matriz_confusion.png")
    model_uri = log_sklearn_model(pipeline, test[features])
    mlflow.set_tag("model_uri", model_uri)

manual_run_id = run.info.run_id
print("run_id:", manual_run_id)
print("Modelo:", model_uri)
display(pd.DataFrame([metrics]).round(3))

**Ahora abrí la interfaz.** Hacé clic en el ícono del experimento (el matraz) en la barra derecha del notebook, o andá a **Experiments → `bigdata_<tu_id>_clase05`**, y entrá al run `01_manual_logistica`. Recorré:

- **Overview / Parameters / Metrics:** lo que registramos en los pasos 1 y 3.
- **Artifacts:** la matriz de confusión.
- **El modelo:** con MLflow 2 aparece como carpeta `model` dentro de *Artifacts*; con MLflow 3 aparece aparte, en **Logged models** (la celda de arriba imprimió una URI `runs:/...` o `models:/m-...` según la versión). En los dos casos guarda `MLmodel` (la descripción del modelo y su firma), el modelo serializado y los requisitos de Python para recrear el entorno.

Mirá la matriz de confusión: con umbral 0,5 el modelo casi no alerta. La accuracy es alta y el recall, bajísimo.

## 3. El mismo run, leído desde código

Todo lo que muestra la interfaz se puede consultar desde código con `MlflowClient`. Es lo que permite automatizar comparaciones y despliegues.

In [ ]:
stored = client.get_run(manual_run_id)
print("Parámetros:", stored.data.params)
print("Métricas:", {k: round(v, 3) for k, v in stored.data.metrics.items()})
print("Artefactos:", [a.path for a in client.list_artifacts(manual_run_id)])

## 4. Autolog: que MLflow registre solo

`mlflow.sklearn.autolog()` intercepta el `fit` de scikit-learn y registra automáticamente parámetros, métricas de entrenamiento y el modelo, sin escribir `log_param` ni `log_metric`. Entrenamos un bosque aleatorio y comparamos con lo que registramos a mano.

In [ ]:
auto_features = FEATURE_SETS["cliente"]
mlflow.sklearn.autolog(silent=True)
with mlflow.start_run(run_name="01_autolog_bosque", tags={**base_tags(config, "01"), "tipo": "autolog"}) as run:
    auto_pipeline = build_pipeline("random_forest", auto_features, n_estimators=100, max_depth=8)
    auto_pipeline.fit(train[auto_features], train["chargeback"])
mlflow.sklearn.autolog(disable=True)

auto = client.get_run(run.info.run_id)
print(f"Parámetros registrados por autolog: {len(auto.data.params)}")
print("Algunos:", dict(list(sorted(auto.data.params.items()))[:8]))
print("Métricas:", {k: round(v, 3) for k, v in auto.data.metrics.items()})
print("Artefactos:", [a.path for a in client.list_artifacts(run.info.run_id)])

**Para observar:** autolog registra *todos* los parámetros del Pipeline (muchos más de los que elegimos) y el modelo, pero sus métricas son **de entrenamiento** (`training_...`): no sabe cuál es nuestro conjunto de test ni que nos importa el PR-AUC. En la práctica se combinan: autolog para no olvidarse de nada y registro manual para lo propio del problema.

Por eso el resto de la práctica usa la función `train_and_log` del helper, que registra siempre lo mismo para que los runs se puedan comparar: los parámetros elegidos, las métricas de test, `train_pr_auc`, tres gráficos, la importancia de cada variable y el modelo con su firma.

## 5. Reproducibilidad

Entrenamos dos veces exactamente la misma configuración. Si el resultado cambia, no podemos confiar en ninguna comparación.

In [ ]:
same = dict(model_type="random_forest", feature_set="cliente", n_estimators=100, max_depth=8, threshold=0.5)
run_a = train_and_log(train, test, config, "01", run_name="01_repro_a", data_version=data_version, **same)
run_b = train_and_log(train, test, config, "01", run_name="01_repro_b", data_version=data_version, **same)
comparison = pd.DataFrame([run_a["metrics"], run_b["metrics"]], index=["01_repro_a", "01_repro_b"]).round(4)
display(comparison)
print("¿Métricas idénticas?", run_a["metrics"] == run_b["metrics"])

Los resultados coinciden porque se fijaron todas las fuentes de variación: **los mismos datos** (`data_version` registra la versión Delta de `ml_features`), **el mismo código**, **la misma semilla** (`random_state=42`) y **el mismo entorno** (el `requirements.txt` que MLflow guarda junto al modelo). Si cambia cualquiera de los cuatro, el run deja de ser reproducible.

## 6. Volver a usar el modelo

El modelo quedó guardado en el run. Cualquiera con acceso al experimento puede cargarlo con su URI y usarlo, sin volver a entrenar.

In [ ]:
loaded = mlflow.sklearn.load_model(model_uri)
sample = test[features].head(5)
display(sample.assign(probabilidad_contracargo=loaded.predict_proba(sample)[:, 1].round(3)))
print(mlflow.models.get_model_info(model_uri).signature)

La **firma** (*signature*) declara qué columnas y tipos espera el modelo y qué devuelve. Si alguien le pasa datos que no cumplen el contrato, MLflow lo frena antes de predecir cualquier cosa:

In [ ]:
generic = mlflow.pyfunc.load_model(model_uri)    # interfaz genérica, la misma para cualquier librería
try:
    generic.predict(sample.drop(columns=["amount"]))
except Exception as error:
    print("Error esperado:", str(error).splitlines()[0][:200])

## Resumen

- Un **run** guarda parámetros, métricas, artefactos y tags; los runs se agrupan en un **experimento**.
- Con clases desbalanceadas la accuracy engaña: usamos **PR-AUC**.
- **Autolog** registra mucho sin código, pero no conoce el problema; el registro manual sí.
- Mismos datos + mismo código + misma semilla + mismo entorno = run **reproducible**.
- El modelo guardado con su **firma** se puede cargar y usar desde cualquier lado.